In [5]:
import locale
import torch

def getpreferredencoding(do_setlocale=True):
    return "utf-8"
locale.getpreferredencoding = getpreferredencoding

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Built with PyTorch: {torch.version.cuda}")
print(f"Is CUDA available? {torch.cuda.is_available()}")

PyTorch Version: 2.12.0+cu132
CUDA Built with PyTorch: 13.2
Is CUDA available? True


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# 1. Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"
DATASET_FILE = "english_to_krakatau_dataset.json"
OUTPUT_DIR = "./krakatau_lora_model"

# 2. Load the Model and Tokenizer
print("Loading model (this might take a minute)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Qwen requires a pad token for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model onto the GPU automatically
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")

# 3. Apply LoRA (The Magic Trick)
# We only train a tiny fraction of the model (q_proj and v_proj)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # You will see it's < 1% of the total model!

# 4. Load and Format Your Dataset
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")

def format_prompt(example):
    # This maps your JSON pairs into exactly what the AI will read
    text = f"Write Java bytecode in Krakatau Jasmin syntax for the following task.\nTask: {example['instruction']}\nCode:\n{example['output']}{tokenizer.eos_token}"
    return {"text": text}

dataset = dataset.map(format_prompt)

# 5. Set up the Trainer
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,   # Lower this to 1 if you run out of GPU memory
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=200,                   # A short test run (increase later)
    dataset_text_field="text",
    max_length=1024,              # Max characters/tokens per example

    # # --- ADD THESE THREE LINES TO FORCE CPU MODE ---
    # use_cpu=True,
    # bf16=False,
    # fp16=False,

    # --- PROGRESS TRACKER SETTINGS ---
    disable_tqdm=False,              
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

# 6. Start Teaching!
print("Starting training loop...")
trainer.train()

# 7. Save the specialized adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training complete! Adapter saved to {OUTPUT_DIR}")

Loading model (this might take a minute)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training loop...


Step,Training Loss
10,0.820708
20,0.562394
30,0.387021
40,0.336010
